# FER2013 Facial Expression Recognition — Experiments Notebook

**Kaggle Competition:** Challenges in Representation Learning: FER2013  
**Task:** 7-class emotion classification from 48×48 grayscale face images  
**Classes:** Angry · Disgust · Fear · Happy · Sad · Surprise · Neutral

This notebook walks through all experiments:
1. Setup (GPU check, packages, data, WandB)
2. Dataset exploration
3. Sanity checks for all architectures
4. Arch 1: TinyMLP (underfitting baseline)
5. Arch 2: PlainCNN (overfitting demonstration)
6. Arch 3: RegCNN + hyperparameter search
7. Arch 4: MiniResNet + hyperparameter search
8. WandB automated sweeps
9. Results analysis

> **Before running:** Set Runtime → Change runtime type → GPU (T4)

## 1. Setup

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install dependencies
!pip install wandb kaggle scikit-learn -q

In [ ]:
# Clone the repository
# Replace with your actual GitHub repo URL
!git clone https://github.com/YOUR_USERNAME/fer-emotion-recognition.git /content/fer
%cd /content/fer

In [ ]:
# Kaggle API setup — upload your kaggle.json
from google.colab import files
files.upload()  # Upload kaggle.json from your machine

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download FER2013 dataset
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q *.zip
!ls -lh fer2013.csv

In [ ]:
# WandB login
import wandb
wandb.login()

## 2. Dataset Exploration

FER2013 contains 35,887 grayscale 48×48 face images split into:
- Training: 28,709 images
- PublicTest (val): 3,589 images  
- PrivateTest (test): 3,589 images

The dataset is **heavily imbalanced** — this is why we use Macro-F1 as the primary metric, not just accuracy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, 'src')

from dataset import EMOTIONS

df = pd.read_csv('fer2013.csv')
print(df.head())
print(f"\nTotal samples: {len(df)}")
print(f"Splits: {df['Usage'].value_counts().to_dict()}")

In [ ]:
# Class distribution — shows imbalance
train_df = df[df['Usage'] == 'Training']
counts = train_df['emotion'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(EMOTIONS, counts.values, color='steelblue')
ax.set_title('Training set class distribution (FER2013)')
ax.set_ylabel('Number of samples')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nMax/Min ratio: {counts.max() / counts.min():.1f}x imbalance")

In [ ]:
# Sample images from each class
fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for col, emotion_idx in enumerate(range(7)):
    subset = train_df[train_df['emotion'] == emotion_idx]
    for row in range(2):
        sample = subset.iloc[row]['pixels']
        img = np.array(sample.split(), dtype=np.uint8).reshape(48, 48)
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(EMOTIONS[emotion_idx], fontsize=9)
plt.suptitle('Sample images per emotion class', y=1.02)
plt.tight_layout()
plt.show()

## 3. Sanity Checks

Before training any architecture we run three checks:

1. **Initial loss check**: At random init, CE loss should be ~ln(7) ≈ 1.946
2. **Overfit one batch**: Loss should reach ~0 in 200 steps (verifies training loop)
3. **Gradient flow**: All layers must have non-zero gradients (verifies backward pass)

These are equivalent to the forward-check and backward-check approach discussed in lectures.

In [ ]:
# Sanity checks for all architectures
for arch in ['tiny', 'plain', 'reg', 'resnet']:
    print(f"\n{'='*60}")
    print(f"SANITY CHECKS — {arch.upper()}")
    print('='*60)
    !python src/sanity_checks.py --csv fer2013.csv --arch {arch}

## 4. Architecture 1: TinyMLP — Underfitting Baseline

```
Flatten → Linear(2304, 128) → ReLU → Linear(128, 7)
~297K parameters
```

**Design decision:** Start with the absolute minimum. No spatial inductive bias (treats pixels as independent), tiny hidden dimension. This will **underfit** — both training and validation accuracy will be low, and the gap between them will be small (both bad).

**Why this matters:** Establishes the floor and motivates CNNs.

In [ ]:
!python src/train.py \
    --csv fer2013.csv \
    --arch tiny \
    --group arch_comparison \
    --name tiny_baseline \
    --epochs 30 \
    --lr 1e-3 \
    --batch 64 \
    --augment false \
    --workers 2

**Expected result:** Val accuracy ≈ 40–45%. Small train-val gap (both low) → underfitting.

The model saturates early and cannot learn further — it simply lacks the capacity and spatial structure to distinguish 7 emotion classes.

## 5. Architecture 2: PlainCNN — Overfitting Demonstration

```
Conv(1→32)→ReLU→Pool → Conv(32→64)→ReLU→Pool → Conv(64→128)→ReLU→Pool
→ Linear(4608, 256) → ReLU → Linear(256, 7)
~1.2M parameters
```

**Design decision:** CNNs are the right architecture for images. But no regularization whatsoever — no BatchNorm, no Dropout, no augmentation. This will **overfit**: training accuracy will be high, but validation accuracy will plateau or decrease.

**Why this matters:** Demonstrates that capacity alone is not enough — regularization is essential.

In [ ]:
!python src/train.py \
    --csv fer2013.csv \
    --arch plain \
    --group arch_comparison \
    --name plain_baseline \
    --epochs 30 \
    --lr 1e-3 \
    --batch 64 \
    --augment false \
    --workers 2

**Expected result:** Train acc ≈ 80–85%, Val acc ≈ 55–60%. Large and growing gap → overfitting.

Look at the WandB charts: val_loss should start increasing while train_loss continues to drop — the classic overfitting signature.

## 6. Architecture 3: RegCNN — Regularized CNN

```
[Conv→BN→ReLU→Conv→BN→ReLU→Pool] × 3  (channels: 64→128→256)
→ Dropout(0.5) → Linear(9216, 512) → ReLU → Dropout(0.5) → Linear(512, 7)
~4.7M parameters
```

**Design decisions:**
- BatchNorm: stabilises training, allows higher LR, mild regularization
- Double conv per block: richer features before downsampling (VGG-style)
- Dropout (0.5): prevents neuron co-adaptation in FC layers
- Data augmentation: increases effective training set size

We explore several hyperparameter settings to understand what matters most.

In [ ]:
# Baseline RegCNN (same group as arch_comparison for fair comparison)
!python src/train.py \
    --csv fer2013.csv \
    --arch reg \
    --group arch_comparison \
    --name reg_baseline \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.5 --wd 1e-4 --augment true \
    --workers 2

In [ ]:
# HP variant: Lower learning rate
# Hypothesis: 1e-4 might be too slow; plateau scheduler might compensate
!python src/train.py \
    --csv fer2013.csv \
    --arch reg \
    --group reg_hp_search \
    --name reg_lr1e-4 \
    --epochs 40 --lr 1e-4 --batch 64 \
    --dropout 0.5 --wd 1e-4 --augment true \
    --workers 2

In [ ]:
# HP variant: Higher dropout
# Hypothesis: 0.6 dropout might over-regularise and cause underfitting
!python src/train.py \
    --csv fer2013.csv \
    --arch reg \
    --group reg_hp_search \
    --name reg_dropout06 \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.6 --wd 1e-4 --augment true \
    --workers 2

In [ ]:
# HP variant: SGD with momentum + cosine annealing
# SGD often generalises better than Adam in the long run
!python src/train.py \
    --csv fer2013.csv \
    --arch reg \
    --group reg_hp_search \
    --name reg_sgd_cosine \
    --epochs 40 --lr 1e-2 --batch 64 \
    --optimizer sgd --dropout 0.5 --wd 5e-4 --augment true \
    --sched cosine \
    --workers 2

In [ ]:
# HP variant: No augmentation (ablation study)
# Shows how much augmentation contributes to generalisation
!python src/train.py \
    --csv fer2013.csv \
    --arch reg \
    --group reg_hp_search \
    --name reg_no_aug \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.5 --wd 1e-4 --augment false \
    --workers 2

## 7. Architecture 4: MiniResNet — Residual Connections + Global Average Pooling

```
Stem: Conv(1→64)→BN→ReLU→Pool
Stage 1-3: StrideConv + ResBlock  (64→128→256→512)
Global Average Pooling → Dropout(0.4) → Linear(512, 7)
~5.3M parameters
```

**Key design decisions:**
- **Skip connections**: Gradient flows through identity path → no vanishing gradient in deeper net
- **Global Average Pooling**: Replaces the huge FC layer (9216→512 in RegCNN). GAP averages each feature map to one number → 512-dim vector, position-invariant, far fewer parameters
- **Label smoothing**: Prevents overconfident predictions, improves calibration

In [ ]:
# Baseline MiniResNet (for arch_comparison group)
!python src/train.py \
    --csv fer2013.csv \
    --arch resnet \
    --group arch_comparison \
    --name resnet_baseline \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.4 --wd 1e-4 --augment true \
    --sched cosine --label_smooth 0.1 \
    --workers 2

In [ ]:
# HP variant: Smaller batch size
# Smaller batches give noisier but sometimes better-generalising gradients
!python src/train.py \
    --csv fer2013.csv \
    --arch resnet \
    --group resnet_hp_search \
    --name resnet_bs32 \
    --epochs 40 --lr 5e-4 --batch 32 \
    --dropout 0.4 --wd 1e-4 --augment true \
    --sched cosine \
    --workers 2

In [ ]:
# HP variant: Plateau scheduler instead of cosine
# Plateau reduces LR only when val_loss stops improving
!python src/train.py \
    --csv fer2013.csv \
    --arch resnet \
    --group resnet_hp_search \
    --name resnet_plateau \
    --epochs 40 --lr 1e-3 --batch 64 \
    --dropout 0.4 --wd 1e-4 --augment true \
    --sched plateau --label_smooth 0.1 \
    --workers 2

## 8. WandB Automated Bayesian Sweeps

Beyond manual HP search, we run automated Bayesian optimization sweeps. The sweep agent samples HP combinations and focuses on the most promising regions of the search space.

In [ ]:
# Initialize RegCNN sweep
!wandb sweep configs/sweep.yaml

In [ ]:
# Run 10 agents for RegCNN sweep
# Replace YOUR_ENTITY and SWEEP_ID with values from the output above
ENTITY = "YOUR_ENTITY"
SWEEP_ID = "PASTE_SWEEP_ID_HERE"

!wandb agent {ENTITY}/fer2013/{SWEEP_ID} --count 10

In [ ]:
# Initialize and run MiniResNet sweep
!wandb sweep configs/sweep_resnet.yaml

## 9. Results Analysis

After all runs complete, compare architectures directly in WandB:
- Go to your WandB project → Group by `arch` or filter by `group=arch_comparison`
- Compare `val_acc`, `f1_macro`, and `val_loss` curves side by side
- Look at the `conf_mat` to see which emotions are confused with each other

In [ ]:
# Pull results from WandB API and plot comparison
import wandb
import pandas as pd
import matplotlib.pyplot as plt

api = wandb.Api()

# Replace YOUR_ENTITY with your WandB username
runs = api.runs("YOUR_ENTITY/fer2013", filters={"group": "arch_comparison"})

results = []
for run in runs:
    summary = run.summary
    results.append({
        'name': run.name,
        'arch': run.config.get('arch'),
        'val_acc': summary.get('val_acc'),
        'test_acc': summary.get('test_acc'),
        'f1_macro': summary.get('f1_macro'),
        'test_f1': summary.get('test_f1'),
    })

results_df = pd.DataFrame(results).sort_values('val_acc', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# Bar chart comparison
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(results_df['arch'], results_df['test_acc'], color='steelblue')
    axes[0].set_title('Test Accuracy by Architecture')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim(0, 1)
    for i, v in enumerate(results_df['test_acc']):
        if v:
            axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')

    axes[1].bar(results_df['arch'], results_df['test_f1'], color='darkorange')
    axes[1].set_title('Test Macro-F1 by Architecture')
    axes[1].set_ylabel('Macro F1')
    axes[1].set_ylim(0, 1)
    for i, v in enumerate(results_df['test_f1']):
        if v:
            axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center')

    plt.tight_layout()
    plt.show()

## Key Takeaways

| Architecture | Behaviour | Root Cause | Fix Applied |
|---|---|---|---|
| TinyMLP | **Underfitting** | No spatial inductive bias, 128 hidden units too small | → Switch to CNN |
| PlainCNN | **Overfitting** | No regularisation, large FC memorises training set | → Add BN, Dropout, augmentation |
| RegCNN | **Good fit** | Regularisation balances capacity and generalisation | → Add residual connections, GAP |
| MiniResNet | **Best** | Skip connections + GAP = deeper + fewer params + position-invariant | Final model |

**On class imbalance:** FER2013 has a 13:1 imbalance (Happy vs Disgust). Macro-F1 reveals this — architectures with high accuracy but low F1 are ignoring minority classes.

**On augmentation:** The ablation run (`reg_no_aug`) shows how much augmentation contributes. Removing it causes val_acc to drop significantly, confirming that data augmentation is not optional for this dataset.